# 08 — Optimisation & Régularisation (DistilBERT)

**Modèle** : `distilbert-base-uncased` fine-tuné sur 10 classes d'intentions client.

**Données** : `../data/client_text_data.pkl` — limité à **50 000 exemples** pour permettre les comparaisons multiples.

**Structure (pattern TP8)** :
```
Partie 1 : Baseline (AdamW, pas de scheduler)
Partie 2 : Comparaison optimiseurs (SGD vs Adam vs AdamW)
Partie 3 : Comparaison schedulers (ReduceLROnPlateau vs CosineAnnealingLR vs Warmup)
Partie 4 : Early stopping
Partie 5 : Weight Decay (régularisation L2)
Bonus    : Grid Search sur LR × weight_decay
```

**Sortie** : `../data/results_optimisation.pkl`, `../models/distilbert_optimised.pt`

## Bloc 1 : Config & imports

In [ ]:
import os, pickle, json, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

DATA_DIR  = '../data/'
MODEL_DIR = '../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_NAME   = 'distilbert-base-uncased'
N_CLASSES    = 10
MAX_LEN      = 128
BATCH_SIZE   = 32
RANDOM_STATE = 42

# Limite à 50K pour permettre les comparaisons multiples en ~10 min par run
# (600K = 5h pour 3 epochs — impossible pour 9+ expériences)
N_SAMPLES = 50000

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'Données utilisées : {N_SAMPLES:,} exemples par split (sous-ensemble de 600K)')

## Bloc 2 : Chargement des données (50K limit)

In [ ]:
import random
random.seed(RANDOM_STATE)

with open(DATA_DIR + 'client_text_data.pkl', 'rb') as f:
    data_full = pickle.load(f)
with open(DATA_DIR + 'label_maps.json') as f:
    label_maps = json.load(f)

# Sous-échantillonnage aléatoire
data = {}
for split in ['train', 'val', 'test']:
    d = data_full[split].copy()
    random.shuffle(d)
    data[split] = d[:N_SAMPLES]

CLIENT_LABELS = {int(k): v for k, v in label_maps['client'].items()}
label_names   = [CLIENT_LABELS[i] for i in range(N_CLASSES)]

print(f'Train : {len(data["train"]):,} | Val : {len(data["val"]):,} | Test : {len(data["test"]):,}')
print(f'(Dataset complet : {len(data_full["train"]):,} train)')

## Bloc 3 : Dataset + DataLoader

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class BertIntentDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.texts     = [d[0] for d in data]
        self.labels    = [d[1] for d in data]
        self.tokenizer = tokenizer
        self.max_len   = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

loaders = {
    split: DataLoader(
        BertIntentDataset(data[split], tokenizer, MAX_LEN),
        batch_size=BATCH_SIZE, shuffle=(split == 'train')
    )
    for split in ('train', 'val', 'test')
}
print(f'Batches train : {len(loaders["train"]):,} | val : {len(loaders["val"]):,}')

## Bloc 4 : Fonctions communes (pattern TP8)

In [ ]:
def build_distilbert():
    """Recrée un DistilBERT frais à chaque expérience pour une comparaison équitable."""
    return DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_CLASSES
    ).to(device)


def train_one_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        # Scheduler batch-level (warmup linéaire)
        if scheduler is not None and not isinstance(scheduler,
            (optim.lr_scheduler.ReduceLROnPlateau, optim.lr_scheduler.CosineAnnealingLR)):
            scheduler.step()
        total_loss += outputs.loss.item() * labels.size(0)
        correct    += (outputs.logits.argmax(dim=1) == labels).sum().item()
        n          += labels.size(0)
    return total_loss / n, correct / n


def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item() * labels.size(0)
            correct    += (outputs.logits.argmax(dim=1) == labels).sum().item()
            n          += labels.size(0)
    return total_loss / n, correct / n


def run_training_full(model, loaders, optimizer, num_epochs=3,
                      scheduler=None, patience=None, label=''):
    """
    Boucle complète (pattern TP8) :
    - scheduler optionnel (epoch-level ou batch-level selon le type)
    - early stopping optionnel (patience)
    - sauvegarde et restauration du meilleur modèle
    """
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc  = 0.0
    best_weights  = copy.deepcopy(model.state_dict())
    no_improve    = 0
    stopped_epoch = None

    for epoch in range(num_epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, loaders['train'], optimizer, scheduler)
        va_loss, va_acc = evaluate(model, loaders['val'])

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)

        # Scheduler epoch-level
        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(va_loss)
            elif isinstance(scheduler, optim.lr_scheduler.CosineAnnealingLR):
                scheduler.step()

        # Sauvegarde du meilleur modèle
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_weights = copy.deepcopy(model.state_dict())
            no_improve   = 0
            flag = ' ✓'
        else:
            no_improve += 1
            flag = f' (patience {no_improve}/{patience})' if patience else ''

        lr = optimizer.param_groups[0]['lr']
        print(f'[{label}] Epoch {epoch+1:2d}/{num_epochs} | '
              f'train {tr_loss:.4f}/{tr_acc:.3f} | '
              f'val {va_loss:.4f}/{va_acc:.3f} | '
              f'LR {lr:.2e} | {time.time()-t0:.0f}s{flag}')

        if patience is not None and no_improve >= patience:
            stopped_epoch = epoch + 1
            print(f'  → Early stopping à epoch {stopped_epoch}')
            break

    history['stopped_epoch'] = stopped_epoch
    history['best_val_acc']  = best_val_acc
    model.load_state_dict(best_weights)
    print(f'  Meilleure val_acc : {best_val_acc:.4f}')
    return model, history


def plot_history(histories, labels, title='Courbes entraînement'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    colors = ['steelblue', 'coral', 'seagreen', 'orchid', 'goldenrod']
    for i, (h, lbl) in enumerate(zip(histories, labels)):
        ep = range(1, len(h['val_loss']) + 1)
        c  = colors[i % len(colors)]
        axes[0].plot(ep, h['train_loss'], '--', color=c, alpha=0.5, label=f'{lbl} train')
        axes[0].plot(ep, h['val_loss'],   '-',  color=c, label=f'{lbl} val')
        axes[1].plot(ep, h['train_acc'],  '--', color=c, alpha=0.5)
        axes[1].plot(ep, h['val_acc'],    '-',  color=c, label=lbl)
        if h.get('stopped_epoch'):
            for ax in axes:
                ax.axvline(h['stopped_epoch'], color=c, linestyle=':', linewidth=0.8)
    for ax, ylabel in zip(axes, ['Loss', 'Accuracy']):
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)
    axes[0].set_title('Loss'); axes[1].set_title('Accuracy')
    plt.suptitle(title); plt.tight_layout(); plt.show()


print('Fonctions chargées : build_distilbert, train_one_epoch, evaluate, run_training_full, plot_history')

## Partie 1 : Baseline

Entraînement le plus simple possible — **AdamW, LR=2e-5, pas de scheduler, pas d'early stopping**.
Cette baseline sert de référence pour mesurer l'apport de chaque technique.

In [ ]:
model_baseline = build_distilbert()
optimizer_baseline = optim.AdamW(model_baseline.parameters(), lr=2e-5, weight_decay=0.01)

print('=== BASELINE : AdamW LR=2e-5, pas de scheduler ===')
model_baseline, history_baseline = run_training_full(
    model_baseline, loaders, optimizer_baseline,
    num_epochs=3, label='Baseline'
)
plot_history([history_baseline], ['Baseline'], 'Baseline — AdamW 2e-5')

## Partie 2 : Comparaison optimiseurs — SGD vs Adam vs AdamW

| Optimiseur | Caractéristique |
|---|---|
| **SGD** | Descente de gradient simple + momentum. Standard pour vision (ResNet). Peu adapté aux transformers. |
| **Adam** | Taux d'apprentissage adaptatif par paramètre. Standard NLP. |
| **AdamW** | Adam + weight decay L2 découplé (Loshchilov 2019). Standard fine-tuning BERT. |

**Pourquoi SGD est-il moins adapté aux transformers ?**
SGD applique le même LR à tous les paramètres. Dans un Transformer, certaines couches (attention heads) ont des gradients de magnitudes très différentes — AdamW les normalise individuellement.

In [ ]:
results_optimiseurs = {}

# ── SGD ──────────────────────────────────────────────────────────────────────
model_sgd = build_distilbert()
# SGD nécessite un LR plus élevé que AdamW (pas d'adaptation par paramètre)
optimizer_sgd = optim.SGD(model_sgd.parameters(), lr=5e-4, momentum=0.9, weight_decay=0.01)
print('=== SGD — LR=5e-4, momentum=0.9 ===')
model_sgd, history_sgd = run_training_full(
    model_sgd, loaders, optimizer_sgd, num_epochs=3, label='SGD'
)
results_optimiseurs['SGD'] = history_sgd['best_val_acc']

# ── Adam ─────────────────────────────────────────────────────────────────────
model_adam = build_distilbert()
optimizer_adam = optim.Adam(model_adam.parameters(), lr=2e-5)
print('\n=== Adam — LR=2e-5 (sans weight decay découplé) ===')
model_adam, history_adam = run_training_full(
    model_adam, loaders, optimizer_adam, num_epochs=3, label='Adam'
)
results_optimiseurs['Adam'] = history_adam['best_val_acc']

# ── AdamW ────────────────────────────────────────────────────────────────────
# Déjà calculé (Baseline)
results_optimiseurs['AdamW'] = history_baseline['best_val_acc']

print('\n=== Résultats comparaison optimiseurs ===')
for name, acc in sorted(results_optimiseurs.items(), key=lambda x: -x[1]):
    print(f'  {name:<8} : val_acc={acc:.4f}')

plot_history(
    [history_sgd, history_adam, history_baseline],
    ['SGD', 'Adam', 'AdamW'],
    'Comparaison optimiseurs — SGD vs Adam vs AdamW'
)

## Partie 3 : Comparaison schedulers

| Scheduler | Principe |
|---|---|
| **Aucun** | LR constant (baseline) |
| **ReduceLROnPlateau** | LR divisé par `factor` si val_loss stagne `patience` epochs |
| **CosineAnnealingLR** | LR descend selon cosinus de LR_max → ~0 sur T_max epochs |
| **Linear Warmup** | LR monte 0→2e-5 sur 10% des steps, descend linéairement ensuite (standard BERT) |

In [ ]:
results_schedulers = {'Aucun (baseline)': history_baseline['best_val_acc']}

# ── ReduceLROnPlateau ─────────────────────────────────────────────────────────
model_plateau = build_distilbert()
opt_plateau   = optim.AdamW(model_plateau.parameters(), lr=2e-5, weight_decay=0.01)
sch_plateau   = optim.lr_scheduler.ReduceLROnPlateau(
    opt_plateau, mode='min', patience=1, factor=0.5
)
print('=== ReduceLROnPlateau (patience=1, factor=0.5) ===')
model_plateau, history_plateau = run_training_full(
    model_plateau, loaders, opt_plateau, num_epochs=3,
    scheduler=sch_plateau, label='Plateau'
)
results_schedulers['ReduceLROnPlateau'] = history_plateau['best_val_acc']

# ── CosineAnnealingLR ─────────────────────────────────────────────────────────
model_cosine = build_distilbert()
opt_cosine   = optim.AdamW(model_cosine.parameters(), lr=2e-5, weight_decay=0.01)
sch_cosine   = optim.lr_scheduler.CosineAnnealingLR(opt_cosine, T_max=3)
print('\n=== CosineAnnealingLR (T_max=3) ===')
model_cosine, history_cosine = run_training_full(
    model_cosine, loaders, opt_cosine, num_epochs=3,
    scheduler=sch_cosine, label='Cosine'
)
results_schedulers['CosineAnnealingLR'] = history_cosine['best_val_acc']

# ── Linear Warmup (standard BERT — Devlin 2018) ───────────────────────────────
model_warmup = build_distilbert()
opt_warmup   = optim.AdamW(model_warmup.parameters(), lr=2e-5, weight_decay=0.01)
num_steps    = len(loaders['train']) * 3
warmup_steps = int(0.1 * num_steps)   # 10% des steps = warmup
sch_warmup   = get_linear_schedule_with_warmup(
    opt_warmup, num_warmup_steps=warmup_steps, num_training_steps=num_steps
)
print(f'\n=== Linear Warmup (warmup={warmup_steps} steps, total={num_steps}) ===')
model_warmup, history_warmup = run_training_full(
    model_warmup, loaders, opt_warmup, num_epochs=3,
    scheduler=sch_warmup, label='Warmup'
)
results_schedulers['Linear Warmup'] = history_warmup['best_val_acc']

print('\n=== Résultats comparaison schedulers ===')
for name, acc in sorted(results_schedulers.items(), key=lambda x: -x[1]):
    print(f'  {name:<22} : val_acc={acc:.4f}')

plot_history(
    [history_baseline, history_plateau, history_cosine, history_warmup],
    ['Aucun', 'ReduceLROnPlateau', 'CosineAnnealingLR', 'Linear Warmup'],
    'Comparaison schedulers'
)

## Partie 4 : Early stopping

L'early stopping arrête l'entraînement quand `val_acc` ne s'améliore plus pendant `patience` epochs
et restaure le meilleur modèle. Évite l'overfitting sans fixer arbitrairement le nombre d'epochs.

> **Différence avec ReduceLROnPlateau** :
> - `ReduceLROnPlateau` : réduit le LR pour continuer à apprendre malgré un plateau
> - Early stopping : *arrête* l'entraînement
> - Les deux sont **complémentaires** et peuvent être combinés

In [ ]:
# Meilleur scheduler de la Partie 3 + early stopping (patience=2)
# On utilise un budget de 6 epochs max — l'early stopping s'arrêtera avant si nécessaire
model_es = build_distilbert()
opt_es   = optim.AdamW(model_es.parameters(), lr=2e-5, weight_decay=0.01)
num_steps_es = len(loaders['train']) * 6
sch_es = get_linear_schedule_with_warmup(
    opt_es,
    num_warmup_steps=int(0.1 * num_steps_es),
    num_training_steps=num_steps_es
)

print('=== Early stopping (patience=2) + Linear Warmup, 6 epochs max ===')
model_es, history_es = run_training_full(
    model_es, loaders, opt_es,
    num_epochs=6, scheduler=sch_es, patience=2, label='EarlyStopping'
)

print(f'\nArrêt à l\'epoch : {history_es["stopped_epoch"]}')
print(f'Meilleure val_acc : {history_es["best_val_acc"]:.4f}')
plot_history([history_es], ['Early Stopping'], 'Early stopping — trait pointillé = arrêt')

## Partie 5 : Weight Decay (régularisation L2)

Le `weight_decay` dans AdamW ajoute une pénalité L2 sur les poids : `loss_total = loss_CE + λ × Σ||w||²`.
Cela pénalise les poids trop grands et force le modèle à apprendre des représentations plus générales.

| weight_decay | Effet |
|---|---|
| 0 | Pas de régularisation — risque d'overfitting |
| 0.01 | Standard BERT — léger, n'entrave pas l'apprentissage |
| 0.1 | Régularisation forte — peut réduire la capacité si trop grand |

In [ ]:
results_wd = {}
histories_wd = []
labels_wd = []

for wd in [0.0, 0.01, 0.1]:
    m = build_distilbert()
    o = optim.AdamW(m.parameters(), lr=2e-5, weight_decay=wd)
    print(f'\n=== weight_decay={wd} ===')
    m, h = run_training_full(m, loaders, o, num_epochs=3, label=f'WD={wd}')
    results_wd[wd] = h['best_val_acc']
    histories_wd.append(h)
    labels_wd.append(f'WD={wd}')

print('\n=== Résultats weight decay ===')
for wd, acc in sorted(results_wd.items(), key=lambda x: -x[1]):
    print(f'  weight_decay={wd:<5} : val_acc={acc:.4f}')

plot_history(histories_wd, labels_wd, 'Impact du Weight Decay')

## Bonus : Grid Search — LR × weight_decay

**Grid Search** : explore toutes les combinaisons d'une grille.
**Random Search** : tire aléatoirement dans un espace continu.

Le Random Search est souvent meilleur que le Grid Search sur de grands espaces
(Bergstra & Bengio 2012) : avec le même budget de runs, il couvre mieux l'espace.

In [ ]:
import itertools

lr_values = [5e-6, 2e-5, 5e-5]
wd_values = [0.0, 0.01, 0.1]

grid = list(itertools.product(lr_values, wd_values))  # 9 combinaisons
print(f'Grid Search : {len(grid)} combinaisons × 3 epochs')

results_grid = []
for lr, wd in grid:
    print(f'\n--- LR={lr:.0e}, WD={wd} ---')
    m = build_distilbert()
    o = optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    _, h = run_training_full(m, loaders, o, num_epochs=3, label=f'LR={lr:.0e},WD={wd}')
    results_grid.append({'lr': lr, 'wd': wd, 'val_acc': h['best_val_acc']})

results_grid.sort(key=lambda x: -x['val_acc'])
print('\n=== Résultats Grid Search (top 5) ===')
for r in results_grid[:5]:
    print(f'  LR={r["lr"]:.0e}  WD={r["wd"]:<5} → val_acc={r["val_acc"]:.4f}')
best = results_grid[0]
print(f'\nMeilleure combinaison : LR={best["lr"]:.0e}, WD={best["wd"]} → {best["val_acc"]:.4f}')

In [ ]:
# Heatmap Grid Search
import numpy as np
import seaborn as sns

matrix = np.zeros((len(lr_values), len(wd_values)))
for r in results_grid:
    i = lr_values.index(r['lr'])
    j = wd_values.index(r['wd'])
    matrix[i, j] = r['val_acc']

plt.figure(figsize=(7, 4))
sns.heatmap(matrix, annot=True, fmt='.4f', cmap='YlGn',
            xticklabels=[str(w) for w in wd_values],
            yticklabels=[f'{lr:.0e}' for lr in lr_values])
plt.xlabel('Weight Decay'); plt.ylabel('Learning Rate')
plt.title('Grid Search — Validation Accuracy')
plt.tight_layout(); plt.show()

## Bloc 11 : Tableau récapitulatif & meilleur modèle

In [ ]:
print('\n' + '='*65)
print(f'{"Expérience":<30} {"Val Acc":>10}')
print('='*65)

all_results = {
    'Baseline (AdamW, no sched)' : history_baseline['best_val_acc'],
    **{f'Optim: {k}': v for k, v in results_optimiseurs.items()},
    **{f'Sched: {k}': v for k, v in results_schedulers.items()},
    'Early stopping (p=2)'       : history_es['best_val_acc'],
    **{f'WD={k}': v for k, v in results_wd.items()},
    f'Grid Best (LR={best["lr"]:.0e},WD={best["wd"]})': best['val_acc'],
}

for name, acc in sorted(all_results.items(), key=lambda x: -x[1]):
    marker = ' ← MEILLEUR' if acc == max(all_results.values()) else ''
    print(f'{name:<30} {acc:>10.4f}{marker}')
print('='*65)

## Bloc 12 : Entraînement final avec les meilleurs hyperparamètres

In [ ]:
# Utiliser les hyperparamètres identifiés dans le Grid Search
BEST_LR = best['lr']
BEST_WD = best['wd']

print(f'Entraînement final : LR={BEST_LR:.0e}, WD={BEST_WD}, warmup, early stopping patience=2')

model_final = build_distilbert()
opt_final   = optim.AdamW(model_final.parameters(), lr=BEST_LR, weight_decay=BEST_WD)
n_steps     = len(loaders['train']) * 5
sch_final   = get_linear_schedule_with_warmup(
    opt_final, num_warmup_steps=int(0.1 * n_steps), num_training_steps=n_steps
)

model_final, history_final = run_training_full(
    model_final, loaders, opt_final,
    num_epochs=5, scheduler=sch_final, patience=2, label='Final'
)

# Evaluation test
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model_final.eval()
preds, targets = [], []
with torch.no_grad():
    for batch in loaders['test']:
        out = model_final(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        preds.extend(out.logits.argmax(dim=1).cpu().tolist())
        targets.extend(batch['label'].tolist())

test_acc = accuracy_score(targets, preds)
report   = classification_report(targets, preds, target_names=label_names)
print(f'\nTest Accuracy (optimisé) : {test_acc:.4f}')
print(f'Test Accuracy (baseline 05) : {history_baseline["best_val_acc"]:.4f}')
print(f'Amélioration : {test_acc - history_baseline["best_val_acc"]:+.4f}')
print(f'\n{report}')

## Bloc 13 : Sauvegarde

In [ ]:
torch.save(model_final.state_dict(), MODEL_DIR + 'distilbert_optimised.pt')

results_optim = {
    'model'            : 'DistilBERT_optimised',
    'best_lr'          : BEST_LR,
    'best_wd'          : BEST_WD,
    'test_accuracy'    : test_acc,
    'baseline_acc'     : history_baseline['best_val_acc'],
    'results_optimiseurs': results_optimiseurs,
    'results_schedulers' : results_schedulers,
    'results_wd'         : results_wd,
    'grid_search'        : results_grid,
    'history_final'      : history_final,
    'predictions'        : preds,
    'targets'            : targets,
}
with open(DATA_DIR + 'results_optimisation.pkl', 'wb') as f:
    pickle.dump(results_optim, f)

print(f'Modèle optimisé → {MODEL_DIR}distilbert_optimised.pt')
print(f'Résultats       → {DATA_DIR}results_optimisation.pkl')